# Taller de Python · Sesión 2
## Pandas con el Titanic: del array a la tabla

**Business Analytics I: Fundamentos y herramientas** · MBDA&IA 2026-27

Al terminar esta sesión sabrás:
1. Cargar una tabla y echarle el primer vistazo.
2. Elegir columnas y filas, y filtrar con condiciones.
3. Limpiar: valores que faltan, tipos incorrectos y filas duplicadas.
4. Crear columnas nuevas y resumir por grupos (`groupby`, `pivot_table`).
5. Explorar los datos y elegir el gráfico adecuado para cada pregunta.

**Hilo conductor:** los 891 pasajeros del Titanic.

> 🟢 **Base**: todos deberían hacerlo. · 🔴 **Reto**: para quien vaya rápido.
> En cada ejercicio, **escribe primero la pregunta como comentario** y después deja que Copilot proponga el código: predice, acepta y comprueba.

⚙️ **Entorno:** abre este notebook en **VS Code**, pulsa *Select Kernel → Colab* y ejecuta la primera celda. Plan B: ábrelo en Colab desde el navegador.

---
## 📋 El encargo

> **Compañía Atlántica de Seguros** · Londres, 6 de mayo de 1912
>
> Le remitimos el **registro completo de 891 pasajeros**. La oficina de Southampton advierte de que hay **casillas en blanco** y alguna ficha copiada dos veces. El comité quiere saber qué perfiles sobrevivieron y si la prensa tenía razón con aquello de **«mujeres y niños primero»**.
>
> *Dirección de Riesgos*

La compañía es ficticia; los pasajeros y sus datos son reales.

## 🤖 Tu becario: Copilot

Hoy el autocompletado de Copilot (el texto en gris) está **activado**. Es un becario rápido y muy seguro de sí mismo, pero sin experiencia: **la IA propone, tú decides**.

| Técnica | Cómo se hace |
|---|---|
| Comentario primero | Escribe la pregunta como comentario (`# tasa de supervivencia por clase`) y deja que Copilot proponga |
| Predice antes de Tab | Antes de aceptar, piensa qué debería salir |
| Palabra a palabra | `Ctrl + →` (Mac: `Cmd + →`) en lugar de `Tab`: lees cada trozo antes de aceptarlo |
| Compara alternativas | Pasa el ratón sobre la sugerencia gris y usa las flechas ‹ 1/3 › para ver otras propuestas; elige una y justifícalo. (Con teclado español, `Alt + ]` no funciona: el `]` pide AltGr.) |
| Comprueba con un dato | Valida cada resultado con una cifra que conozcas: 891 filas, 177 edades vacías, 74 % de mujeres… |

- Al **Chat** de Copilot se le pide «explícame este error» o «¿hay otra forma?», nunca «hazlo». El plan gratuito tiene 50 mensajes de chat al mes: resérvalos para errores que no entiendas.
- **Si no puedes explicar una línea, no la entregas.**
- Al final hay una **bitácora de IA**: apunta qué te propuso, qué aceptaste, qué cambiaste y por qué.

---
# 0. Repaso: NumPy con los 891

La sesión pasada vimos NumPy con prisa. Repasamos ahora con el registro completo. La celda siguiente carga tres arrays: `edades`, `tarifas` y `sobrevivio`. **No te preocupes por cómo**: es justo lo que aprenderemos en la §1.

Un **array** es como una **cubitera**: todos los huecos iguales, con lo mismo dentro, y se llena de una vez. Una lista de Python es más bien una caja de mudanza: cabe de todo, pero cada cosa se saca por separado.

> En cada celda, **predice primero** qué va a salir y después ejecuta.

In [ ]:
import numpy as np
import pandas as pd

URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
_df = pd.read_csv(URL)                      # (lo veremos en la §1)
edades = _df["Age"].to_numpy()
tarifas = _df["Fare"].to_numpy()
sobrevivio = _df["Survived"].to_numpy()
print(len(edades), "pasajeros ·", edades.dtype, tarifas.dtype, sobrevivio.dtype)

**Vectorizar.** ¿Cuántas líneas hacen falta para subir un 3 % las 891 tarifas?

In [ ]:
nuevas = tarifas * 1.03          # una línea, sin bucle: la cinta transportadora
print(tarifas[:5])
print(nuevas[:5].round(2))

**Media y mediana.** ¿Saldrán parecidas o muy distintas?

In [ ]:
print(f"Media: {np.mean(tarifas):.1f} · Mediana: {np.median(tarifas):.1f} · Máximo: {tarifas.max():.1f}")

Unos pocos billetes muy caros (hasta 512 libras) tiran de la media. Para describir al pasajero «típico», la **mediana** es más honesta.

**Datos que faltan.** Hay 177 edades vacías. ¿Qué pasará con la media?

In [ ]:
print("Vacías:", np.isnan(edades).sum())
print("mean:", np.mean(edades))              # nan: un hueco contamina el resultado
print("nanmean:", round(np.nanmean(edades), 1))

**Máscaras (el colador).** Si sobrevivió el 38 % del total, ¿más o menos entre los niños?

In [ ]:
ninos = edades < 16                          # True / False para cada pasajero
print("Niños:", ninos.sum())
print(f"Supervivencia global: {sobrevivio.mean():.0%} · niños: {sobrevivio[ninos].mean():.0%}")

⚠️ Una edad vacía (`nan`) da `False` en `edades < 16`: el colador la trata como adulto sin que nadie lo haya decidido. Lo retomaremos.

🟢 **Ejercicio 0.1** · ¿Cuántos pasajeros pagaron **más de 100 libras** y qué porcentaje sobrevivió?

In [ ]:
# 1. Escribe aquí la pregunta como comentario:
# 
# 2. Deja que Copilot proponga el código: predice, acepta y comprueba con un dato.

🔴 **Reto 0.2** · ¿Qué proporción de las tarifas está **por encima de la media**? ¿Qué te dice eso de la media?

In [ ]:
# 1. Escribe aquí la pregunta como comentario:
# 
# 2. Deja que Copilot proponga el código: predice, acepta y comprueba con un dato.

---
# 1. Series y DataFrames: el archivador de fichas

> 📨 **Encargo:** «Ha llegado el registro completo. Antes de calcular nada, compruebe qué contiene.»

- Un **DataFrame** es la lista de diccionarios de la sesión 1, ya ordenada como tabla y con las columnas rotuladas: un **archivador de fichas**.
- Una **Series** es un **cajón** del archivador: una sola columna, con el número de ficha (el **índice**) junto a cada valor. Por debajo es un array de NumPy con etiquetas.

In [ ]:
df = pd.read_csv(URL)    # una línea frente a toda la §9 de la sesión 1
df.head()

In [ ]:
df.shape                 # (filas, columnas)

In [ ]:
df.info()                # columnas, tipos y cuántos valores hay en cada una

Fíjate en `Non-Null Count`: **Age**, **Cabin** y **Embarked** tienen huecos. Lo resolveremos en la §3.

In [ ]:
df[["Age", "Fare"]].describe().round(1)    # resumen de las columnas numéricas

### Una Series: un cajón del archivador

In [ ]:
edad = df["Age"]
print(type(edad))
print(edad.head())
print("Media:", round(edad.mean(), 1))     # Pandas ignora los nulos por defecto (el nanmean de NumPy)

In [ ]:
df["Embarked"].value_counts()              # cuántos de cada valor: S = Southampton, C = Cherburgo, Q = Queenstown

In [ ]:
df["Fare"].to_numpy()[:5]                  # por debajo, el array de NumPy del repaso

🤖 **Momento IA** · Quita el `#` de la línea siguiente y ejecuta. Copia el error en el Chat de Copilot y pídele: «explícame este error». Anótalo en la bitácora.

In [ ]:
# df["age"]

> Python distingue mayúsculas: la columna se llama `Age`, no `age` (`KeyError`).

🟢 **Ejercicio 1.1** · Técnica **comentario primero**: escribe solo el comentario `# tarifa media y tarifa máxima` y deja que Copilot proponga. Antes de aceptar, **predice** qué debería salir (repaso: 32,2 y 512,3).

In [ ]:
# tarifa media y tarifa máxima

🔴 **Reto 1.2** · ¿Cuántos pasajeros viajaban en cada clase?

In [ ]:
# 1. Escribe aquí la pregunta como comentario:
# 
# 2. Deja que Copilot proponga el código: predice, acepta y comprueba con un dato.

---
# 2. Selección y filtrado: el colador con 891 filas

> 📨 **Encargo:** «La Dirección pide solo nombre, sexo y edad de algunos expedientes, y ver solo ciertos perfiles.»

| Quiero… | Se escribe | Devuelve |
|---|---|---|
| Una columna | `df["Sex"]` | Series |
| Varias columnas | `df[["Name", "Sex", "Age"]]` | DataFrame |
| Una fila por su índice | `df.loc[2]` | Series |
| Filas y columnas | `df.loc[0:2, ["Name", "Age"]]` | DataFrame (con `loc`, el 2 **entra**) |
| Filas que cumplen algo | `df[df["Age"] < 30]` | DataFrame |

In [ ]:
df[["Name", "Sex", "Age"]].head(3)

In [ ]:
df.loc[0:2, ["Name", "Age"]]

### Filtrar con máscaras
Es el colador de la sesión 1, ahora sobre todas las filas. Para combinar condiciones se usan `&` (y), `|` (o) y `~` (no), **siempre con paréntesis**.

In [ ]:
mujeres = df[df["Sex"] == "female"]
print(len(mujeres), "mujeres")

tercera_y_mujer = df[(df["Sex"] == "female") & (df["Pclass"] == 3)]
print(len(tercera_y_mujer), "mujeres de 3ª")

mujer_o_nino = df[(df["Sex"] == "female") | (df["Age"] < 16)]
print(len(mujer_o_nino), "mujeres o niños")

no_sobrevivio = df[~(df["Survived"] == 1)]
print(len(no_sobrevivio), "no sobrevivieron")

In [ ]:
df[df["Embarked"].isin(["C", "Q"])].shape     # isin: ¿está en esta lista?

In [ ]:
df.query("Pclass == 1 and Age < 18").shape    # query: lo mismo escrito como frase (solo para leer)

🤖 **Momento IA** · Quita el `#` y ejecuta. Pide al Chat que te explique el error y te proponga la alternativa. **Comprueba** que el resultado cuadra con la celda anterior (144 mujeres de 3ª).

In [ ]:
# df[(df["Sex"] == "female") and (df["Pclass"] == 3)]

> `and` espera un único `True` o `False`; con columnas enteras hay que usar `&`.

🟢 **Ejercicio 2.1** · Técnica **predice antes de Tab**. ¿Cuántas **mujeres de 3ª clase** sobrevivieron? Antes de aceptar la sugerencia de Copilot, di en voz alta qué tiene que aparecer en el código.

In [ ]:
# 1. Escribe aquí la pregunta como comentario:
# 
# 2. Deja que Copilot proponga el código: predice, acepta y comprueba con un dato.

🔴 **Reto 2.2** · Pasajeros que embarcaron en **Cherburgo** (`C`) con tarifa **superior a la mediana** de todo el barco. ¿Cuántos son y qué porcentaje sobrevivió?

In [ ]:
# 1. Escribe aquí la pregunta como comentario:
# 
# 2. Deja que Copilot proponga el código: predice, acepta y comprueba con un dato.

---
# 3. Limpieza: nulos, tipos y duplicados

> 📨 **Encargo:** «No firmamos nada con datos sucios.»

Un nulo es una **casilla en blanco** en un formulario. Antes de rellenarla hay que decidir **qué significa** y **dejar escrita la decisión**.

In [ ]:
df.isna().sum()

| Columna | Huecos | Decisión |
|---|---|---|
| `Embarked` | 2 | Rellenar con el puerto más frecuente (impacto mínimo) |
| `Age` | 177 | Rellenar con la mediana **de su clase** y marcar que es una estimación |
| `Cabin` | 687 | No se rellena: se crea `tiene_cabina` (sí / no) |

Primero, lo que **no** hay que hacer sin pensar:

In [ ]:
sin_nulos = df.dropna()
print(f"Quedan {len(sin_nulos)} de {len(df)} filas")
print(sin_nulos["Pclass"].value_counts().sort_index())
print(f"Supervivencia: {df['Survived'].mean():.0%} → {sin_nulos['Survived'].mean():.0%}")

Basta un hueco en **cualquier** columna (Cabin tiene 687) para borrar la fila: quedan 183 pasajeros, casi todos de 1ª clase, y la supervivencia «sube». **El análisis cambia de historia.**

Hacemos la limpieza en una **copia**, `df_limpio`. Guardamos `df` con las edades reales: más adelante no queremos que una estimación decida si alguien era niño.

In [ ]:
df_limpio = df.copy()

# Embarked: el puerto más frecuente
puerto_frecuente = df_limpio["Embarked"].mode()[0]
df_limpio["Embarked"] = df_limpio["Embarked"].fillna(puerto_frecuente)

# Age: anotar qué edades son estimadas y rellenar con la mediana de su clase
df_limpio["edad_estimada"] = df_limpio["Age"].isna()
print(df_limpio.groupby("Pclass")["Age"].median())
df_limpio["Age"] = df_limpio["Age"].fillna(df_limpio.groupby("Pclass")["Age"].transform("median"))

# Cabin: no se rellena; se deriva una columna sí / no
df_limpio["tiene_cabina"] = df_limpio["Cabin"].notna()

df_limpio.isna().sum()

### Tipos y duplicados: el registro de Southampton
La oficina de Southampton ha enviado **su copia** del registro. Veamos qué llega.

In [ ]:
URL_SUCIO = "https://raw.githubusercontent.com/javialani/taller-python-mbda-2026/main/datos/titanic_sucio.csv"
sucio = pd.read_csv(URL_SUCIO)
sucio.info()

In [ ]:
sucio[["Name", "Pclass", "Age", "Fare"]].head()

Problemas: `Pclass` llega como `"3ª"`, `Age` a veces como `"38 años"` y `Fare` con coma decimal (`"7,25"`). Todo es **texto** (`object` o `str`): es el `"22" + 1` de la sesión 1. Además hay fichas repetidas.

🤖 Técnica **palabra a palabra**: en las celdas de limpieza, acepta las sugerencias de Copilot con `Ctrl + →` y lee cada trozo antes de aceptarlo.

In [ ]:
s = sucio.copy()
s["Age"] = pd.to_numeric(s["Age"].str.replace(" años", ""), errors="coerce")   # lo que no se puede convertir → NaN
s["Fare"] = pd.to_numeric(s["Fare"].str.replace(",", "."))
s["Pclass"] = s["Pclass"].str.replace("ª", "").astype(int).astype("category")   # la clase es una etiqueta, no una cantidad
s.dtypes[["Pclass", "Age", "Fare"]]

In [ ]:
print("Duplicados:", s.duplicated().sum())
s = s.drop_duplicates()
print("Filas tras quitarlos:", len(s))

🟢 **Ejercicio 3.1** · Comprueba con un dato que la limpieza ha funcionado: ¿coinciden ahora el número de filas, la edad media y la tarifa media de `s` con las de `df`?

In [ ]:
# 1. Escribe aquí la pregunta como comentario:
# 
# 2. Deja que Copilot proponga el código: predice, acepta y comprueba con un dato.

🔴 **Reto 3.2** · Calcula la tarifa media de `sucio` **antes** de quitar los duplicados (convirtiendo solo `Fare`). ¿Cambia el resultado? ¿Por qué importa el orden «primero limpiar, después resumir»?

In [ ]:
# 1. Escribe aquí la pregunta como comentario:
# 
# 2. Deja que Copilot proponga el código: predice, acepta y comprueba con un dato.

---
# 4. Transformación y agregación

> 📨 **Encargo:** «Prepare una tabla de supervivencia por perfil para el comité de tarificación.»

### Columnas derivadas

In [ ]:
df["familia"] = df["SibSp"] + df["Parch"] + 1        # hermanos/cónyuge + padres/hijos + el propio pasajero
df["grupo_edad"] = pd.cut(df["Age"], bins=[0, 16, 30, 60, 100],
                          labels=["niño", "joven", "adulto", "mayor"], right=False)
df["es_nino"] = df["Age"] < 16
df[["Name", "Age", "familia", "grupo_edad", "es_nino"]].head(6)

Mira a Mr. Moran (fila 5): edad desconocida, así que su `grupo_edad` es `NaN`… pero `es_nino` es `False`. ¿Es un adulto o no lo sabemos?

### `groupby`: separar en buzones y calcular un dato por grupo
Como clasificar el correo: **separar** en buzones, **calcular un dato** de cada buzón (cuántas cartas hay, cuánto pesan) y **combinar** los resultados en una hoja.

In [ ]:
df.groupby("Sex")["Survived"].mean()                  # la media de 0 y 1 es una tasa

In [ ]:
df.groupby("Pclass")["Survived"].agg(["mean", "size"])   # muestra siempre el tamaño junto a la tasa

In [ ]:
tramos_familia = pd.cut(df["familia"], bins=[0, 1, 4, 11], labels=["solo", "2 a 4", "5 o más"])
df.groupby(tramos_familia, observed=True)["Survived"].agg(["mean", "size"]).round(2)

Ni solos ni en familias grandes: la relación **no es lineal**.

### `pivot_table`: dos criterios a la vez

In [ ]:
df.pivot_table(values="Survived", index="Sex", columns="Pclass", aggfunc="mean").round(2)

In [ ]:
df.pivot_table(values="Survived", index="Sex", columns="Pclass", aggfunc="size")    # ¿cuántos hay en cada casilla?

🤖 Técnica **compara alternativas**: escribe el comentario `# supervivencia por sexo y clase con groupby` y recorre las propuestas con las flechas ‹ 1/3 › que aparecen al pasar el ratón sobre la sugerencia. Una opción habitual es `groupby` + `unstack`. **Comprueba** que da los mismos números:

In [ ]:
a = df.pivot_table(values="Survived", index="Sex", columns="Pclass", aggfunc="mean")
b = df.groupby(["Sex", "Pclass"])["Survived"].mean().unstack()
print(np.allclose(a, b))      # True: dos caminos, mismo resultado

🟢 **Ejercicio 4.1** · Tasa de supervivencia y número de pasajeros por **puerto de embarque**.

In [ ]:
# 1. Escribe aquí la pregunta como comentario:
# 
# 2. Deja que Copilot proponga el código: predice, acepta y comprueba con un dato.

🔴 **Reto 4.2** · Tasa por `grupo_edad` **y** clase, con el tamaño de cada grupo. ¿Qué grupo es demasiado pequeño para sacar conclusiones?

In [ ]:
# 1. Escribe aquí la pregunta como comentario:
# 
# 2. Deja que Copilot proponga el código: predice, acepta y comprueba con un dato.

---
# 5. EDA y visualización

> 📨 **Encargo:** «Que se entienda en una diapositiva: el comité no lee tablas.»

El **EDA** (análisis exploratorio) es la **primera consulta médica**: tomar las constantes vitales antes de diagnosticar.

| Constante | Pregunta | Herramienta |
|---|---|---|
| Tamaño | ¿Cuánto hay? | `df.shape` |
| Huecos | ¿Qué falta? | `df.isna().sum()` |
| Reparto | ¿Cómo se distribuye cada variable? | `df.describe()`, histogramas |
| Relaciones | ¿Qué se mueve junto? | `df.corr(numeric_only=True)`, mapa de calor |

### Un gráfico para cada pregunta
- **Repartir** (¿cómo se distribuye?) → histograma.
- **Comparar** (¿qué grupo tiene más?) → barras, con el eje desde 0.
- **Relacionar** (¿qué se mueve junto?) → mapa de calor o dispersión.

Y un **título que diga la conclusión**, no el nombre de la columna.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

plt.hist(df["Age"].dropna(), bins=8, color="#0A8F86")
plt.title("La mayoría tenía entre 20 y 40 años")
plt.xlabel("Edad (años)")
plt.ylabel("Pasajeros")
plt.show()

In [ ]:
sns.countplot(data=df, x="Sex", hue="Survived")
plt.title("Murieron muchos más hombres que mujeres")
plt.show()

In [ ]:
sns.barplot(data=df, x="Pclass", y="Survived", hue="Sex", errorbar=None)
plt.title("La clase importa, y el sexo aún más")
plt.ylabel("Tasa de supervivencia")
plt.show()

In [ ]:
sns.histplot(data=df, x="Age", hue="Survived", multiple="stack", bins=16)
plt.title("Los niños pequeños sobrevivieron más")
plt.show()

In [ ]:
corr = df[["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="BrBG", center=0)
plt.title("Sobrevivir va con la clase y la tarifa")
plt.show()

`Pclass` y `Survived`: −0,34, porque «clase 1» es la mejor. El **sexo no aparece**: es texto, y la correlación solo ve números. La correlación no lo ve todo.

🟢 **Ejercicio 5.1** · Haz **el gráfico que llevarías a la Dirección** para responder: «¿influyó el puerto de embarque?». Título con la conclusión.

In [ ]:
# 1. Escribe aquí la pregunta como comentario:
# 
# 2. Deja que Copilot proponga el código: predice, acepta y comprueba con un dato.

🔴 **Reto 5.2** · Este gráfico engaña. ¿Por qué? Arréglalo.

In [ ]:
sns.barplot(data=df, x="Pclass", y="Survived", errorbar=None)
plt.ylim(0.2, 0.65)
plt.title("¿Cuánto más sobrevivió la 1ª clase?")
plt.show()

In [ ]:
# 1. Escribe aquí la pregunta como comentario:
# 
# 2. Deja que Copilot proponga el código: predice, acepta y comprueba con un dato.

---
# 6. Auditar al becario y reto final

> 📨 **Encargo:** «El becario ha dejado varias celdas hechas. Revíselas antes de que lleguen al comité.»

Cada propuesta de Copilot se **ejecuta sin errores**. ¿Qué tiene de malo para el negocio? Piénsalo antes de leer la respuesta.

| Comentario | Propuesta del becario |
|---|---|
| `# limpia los nulos del dataset` | `df = df.dropna()` |
| `# rellena las edades vacías` | `df["Age"] = df["Age"].fillna(df["Age"].mean())` |
| `# marca a los niños` | `df["es_nino"] = df["Age"] < 16` |
| `# tarifa media del registro de Southampton` | `sucio["Fare"]` convertida y promediada sin quitar duplicados |

✍️ **Tu diagnóstico** (escribe aquí, en una frase cada una):

1.
2.
3.
4.

## 🏁 Reto: ¿fue real el «mujeres y niños primero»?

> 📨 **Encargo:** «La prensa dice que se cumplió. ¿Lo confirman los datos?»

Entrega **una tabla, un gráfico y dos frases de conclusión**. Niño: menor de 16 años. Usamos `df` (edades reales): las 177 edades desconocidas quedan fuera de niños y de adultos.

🟢 **Base** · Compara la supervivencia de **mujeres**, **niños** y **hombres adultos**.

In [ ]:
# 1. Escribe aquí la pregunta como comentario:
# 
# 2. Deja que Copilot proponga el código: predice, acepta y comprueba con un dato.

🔴 **Reto** · Cruza el perfil con la **clase**. ¿Fue igual para todos?

In [ ]:
# 1. Escribe aquí la pregunta como comentario:
# 
# 2. Deja que Copilot proponga el código: predice, acepta y comprueba con un dato.

In [ ]:
# 1. Escribe aquí la pregunta como comentario:
# 
# 2. Deja que Copilot proponga el código: predice, acepta y comprueba con un dato.

✍️ **Tus dos frases para la Dirección:**

1.
2.

---
## Resumen de hoy
| Concepto | Símil o idea clave |
|---|---|
| DataFrame / Series | Archivador de fichas / un cajón con su índice |
| `read_csv`, `head`, `info`, `describe` | Cargar y echar el primer vistazo |
| `df["col"]`, `df[[...]]`, `loc` | Elegir columnas, filas o las dos |
| `df[mascara]`, `& \| ~` | El colador con 891 filas |
| `isna`, `fillna`, `dropna` | Casillas en blanco: decidir y anotar |
| `to_numeric`, `astype`, `drop_duplicates` | Tipos correctos y sin copias |
| `groupby` | Separar en buzones y calcular un dato por grupo |
| `pivot_table` | Dos criterios a la vez |
| EDA | La primera consulta médica |
| `hist`, `barplot`, `heatmap` | Repartir, comparar, relacionar |
| Copilot | El becario propone, tú decides |

## 📓 Bitácora de IA
| Sección | Qué te propuso Copilot (o qué le preguntaste) | ¿Aceptado, modificado o rechazado? | Por qué / cómo lo comprobaste |
|---|---|---|---|
| §1 | | | |
| §2 | | | |
| §3 | | | |
| §4 | | | |
| §5 | | | |
| §6 | | | |

## ✅ Autoevaluación
1. ¿Por qué `df["Age"].mean()` no da `nan`?
2. ¿Qué diferencia hay entre `df["Sex"]` y `df[["Sex"]]`?
3. ¿Por qué falla `df[cond1 and cond2]`?
4. ¿Por qué no basta con `df.dropna()`?
5. ¿Qué tres pasos hace un `groupby`?
6. Copilot propone un código que funciona. ¿Ya está?

**Para la sesión 3:** trae elegido el dataset de tu proyecto. Los datos de una empresa suelen venir en varias tablas: las uniremos con `merge`.